In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [2]:
# Load dataset
df = pd.read_csv("PhiUSIIL_Phishing_URL_Dataset.csv")

# Drop non-numeric columns
df = df.drop(columns=["FILENAME", "URL", "Domain", "Title", "TLD"])

# Separate features and labels
X = df.drop(columns=["label"])
y = df["label"]


In [ ]:
import numpy as np

data_clean = df.select_dtypes(include=[np.number]).dropna()

#set target 
X = data_clean.drop(columns=['label'])
y = data_clean['label']

#shows correlation between two features that is above .9 
corr = X.corr()
high_corr = [(col1, col2) for col1 in corr.columns 
             for col2 in corr.columns 
             if col1 < col2 and abs(corr[col1][col2]) > 0.9]

for pair in high_corr:
    print(f"{pair[0]} - {pair[1]}")

    # check how correlated each feature is with the label
# high correlation with the label = potential data leakage
# the model would be learning from features that essentially encode the answer
label_corr = data_clean.corr()['label'].drop('label')
print(label_corr.abs().sort_values(ascending=False).to_string())

# drop one from each highly inter-correlated pair
cols_to_drop_corr = [pair[0] for pair in high_corr]
print(f"Dropping correlated features: {cols_to_drop_corr}")
X = X.drop(columns=cols_to_drop_corr)

# drop features with label correlation (data leakage)
threshold = 0.25 #this was all trial and error, this threshold is not statistically calculated
leaky_cols = label_corr[label_corr.abs() >= threshold].index.tolist()
leaky_cols = [col for col in leaky_cols if col in X.columns]

print(f"Dropping leaky features: {leaky_cols}")
X = X.drop(columns=leaky_cols)

print(f"\nRemaining features ({X.shape[1]}): {list(X.columns)}")


NoOfLettersInURL - URLLength
DomainTitleMatchScore - URLTitleMatchScore
URLSimilarityIndex            0.860358
HasSocialNet                  0.784255
HasCopyrightInfo              0.743358
HasDescription                0.690232
IsHTTPS                       0.609132
DomainTitleMatchScore         0.584905
HasSubmitButton               0.578561
IsResponsive                  0.548608
URLTitleMatchScore            0.539419
SpacialCharRatioInURL         0.533537
HasHiddenFields               0.507731
HasFavicon                    0.493711
URLCharProb                   0.469749
CharContinuationRate          0.467735
HasTitle                      0.459725
DegitRatioInURL               0.432032
Robots                        0.392620
NoOfJS                        0.373500
LetterRatioInURL              0.367794
Pay                           0.359747
NoOfOtherSpecialCharsInURL    0.358891
NoOfSelfRef                   0.316211
DomainLength                  0.283152
NoOfImage                     0

In [23]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Decision Tree

In [27]:
# -------- Decision Tree --------
dt = DecisionTreeClassifier(
    max_depth=2,           # Very shallow (same as Random Forest)
    min_samples_split=50,  # Require many samples to split
    min_samples_leaf=25,   # Require many in each leaf
    random_state=42
)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))


Decision Tree Accuracy: 0.9449945927606608
              precision    recall  f1-score   support

           0       0.96      0.91      0.93     20124
           1       0.94      0.97      0.95     27035

    accuracy                           0.94     47159
   macro avg       0.95      0.94      0.94     47159
weighted avg       0.95      0.94      0.94     47159



# Random Forest

In [ ]:
# -------- Random Forest --------
rf = RandomForestClassifier(
    n_estimators=10,      # Very few trees
    max_depth=2,          # Extremely shallow
    min_samples_split=50, # Require many samples to split
    min_samples_leaf=25,  # Require many in each leaf
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.9278610657562714
              precision    recall  f1-score   support

           0       0.96      0.87      0.91     20124
           1       0.91      0.97      0.94     27035

    accuracy                           0.93     47159
   macro avg       0.93      0.92      0.93     47159
weighted avg       0.93      0.93      0.93     47159



# Neural Network

In [28]:
# -------- Neural Network --------
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Scale the features for neural network (important for NN performance)
scaler_nn = StandardScaler()
X_train_scaled = scaler_nn.fit_transform(X_train)
X_test_scaled = scaler_nn.transform(X_test)

# Create and train neural network
# Using a simple 2-layer architecture to avoid overfitting
nn = MLPClassifier(
    hidden_layer_sizes=(32, 16),  # Two hidden layers: 32 and 16 neurons
    max_iter=500,                  # Max iterations for training
    alpha=0.01,                    # L2 regularization strength (prevents overfitting)
    learning_rate_init=0.001,      # Learning rate
    early_stopping=True,           # Stop when validation score plateaus
    validation_fraction=0.1,       # Use 10% of training data for validation
    random_state=42
)

nn.fit(X_train_scaled, y_train)

y_pred_nn = nn.predict(X_test_scaled)

print("Neural Network Accuracy:", accuracy_score(y_test, y_pred_nn))
print(classification_report(y_test, y_pred_nn))


Neural Network Accuracy: 0.9805339383786764
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     20124
           1       0.98      0.98      0.98     27035

    accuracy                           0.98     47159
   macro avg       0.98      0.98      0.98     47159
weighted avg       0.98      0.98      0.98     47159



In [29]:
# -------- Model Comparison --------
import pandas as pd

# Create a comparison dataframe
accuracies = {
    'Decision Tree': accuracy_score(y_test, y_pred_dt),
    'Random Forest': accuracy_score(y_test, y_pred_rf),
    'Neural Network': accuracy_score(y_test, y_pred_nn)
}

comparison_df = pd.DataFrame(list(accuracies.items()), columns=['Model', 'Accuracy'])
comparison_df = comparison_df.sort_values('Accuracy', ascending=False)

print("\n=== MODEL COMPARISON ===")
print(comparison_df.to_string(index=False))
print(f"\nBest Model: {comparison_df.iloc[0]['Model']} with {comparison_df.iloc[0]['Accuracy']:.4f} accuracy")




=== MODEL COMPARISON ===
         Model  Accuracy
Neural Network  0.980534
 Decision Tree  0.944995
 Random Forest  0.927861

Best Model: Neural Network with 0.9805 accuracy
